# Fine-tuning the four Arabic language models

This notebook fine-tunes one base model at a time on the revised context–gloss dataset. The four configurations correspond to AraBERT, CAMeLBERT-MSA, ARBERTv2 and Arabic ALBERT.

Running all four configurations requires a CUDA-capable environment. Select a model with `MODEL_KEY`, run the notebook, then repeat for the remaining configurations. Each run saves positive-class probabilities for the common validation and test sets; those files are consumed by the stacking notebook.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

from arabic_wsd.experiments import MODEL_CONFIGS, classification_metrics, load_pairs

## Configuration

The training settings follow the experiment notebook: learning rate `2e-5`, batch size `16`, linear scheduling, 10% warmup and model selection on the validation set. ARBERTv2 used two epochs; the other configurations used four.

In [ ]:
MODEL_KEY = 'arabert'  # arabert, camelbert, arbertv2, or arabic-albert
SEED = 9
MAX_LENGTH = 512

config = MODEL_CONFIGS[MODEL_KEY]
config

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'arabic_verb_context_gloss_pairs_revised.xlsx'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / MODEL_KEY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pairs = load_pairs(DATA_PATH)
dataset = Dataset.from_pandas(pairs, preserve_index=False)
train_and_temporary = dataset.train_test_split(test_size=0.20, seed=SEED)
validation_and_test = train_and_temporary['test'].train_test_split(test_size=0.50, seed=SEED)
splits = DatasetDict({
    'train': train_and_temporary['train'],
    'validation': validation_and_test['train'],
    'test': validation_and_test['test'],
})

## Paired-sequence tokenization

The context and candidate gloss are passed as two sequences. This preserves the structure of the WSD decision instead of concatenating them into an undocumented free-text format.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.checkpoint)

def tokenize_batch(batch):
    return tokenizer(
        batch['verb+context'],
        batch['sense'],
        max_length=MAX_LENGTH,
        truncation=True,
    )

tokenized = splits.map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(evaluation):
    logits, labels = evaluation
    predictions = np.argmax(logits, axis=-1)
    return classification_metrics(labels, predictions)

## Fine-tuning and held-out evaluation

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(config.checkpoint, num_labels=2)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=config.epochs,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    lr_scheduler_type='linear',
    warmup_ratio=0.1,
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
test_metrics = trainer.evaluate(tokenized['test'])
test_metrics

## Save the model and stacking features

The probability assigned to label 1 becomes one input feature for the stacking model. Labels are saved alongside the probabilities to make alignment checks explicit.

In [ ]:
trainer.save_model(str(OUTPUT_DIR / 'model'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'model'))

for split_name in ('validation', 'test'):
    prediction_output = trainer.predict(tokenized[split_name])
    probabilities = torch.softmax(torch.tensor(prediction_output.predictions), dim=-1).numpy()[:, 1]
    np.savez_compressed(
        OUTPUT_DIR / f'{split_name}_predictions.npz',
        positive_probability=probabilities,
        label=prediction_output.label_ids,
    )